# EnMP banana

**En**hance **M**y **P**laylist **banana** — SpotifyプレイリストURLから縦サムネ(1080x1920, IGストーリー規格)を作るColabノート。

- Spotifyへのユーザー連携は不要（Client Credentials Flowでアプリ登録の鍵だけ使う）
- 背景アート: Gemini ( `gemini-2.5-flash-image` / `gemini-3-pro-image-preview` ) または OpenAI ( `gpt-image-1` )
- タイトル・作者・Spotifyロゴ・"Let's listen on Spotify!" は PIL で合成するので崩れない


## 1. セットアップ
リポジトリを取得して依存をインストールする。日本語フォントも入れておく。

In [ ]:
import os, sys
REPO_DIR = "/content/EnMP-banana"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone https://github.com/Rhizobium-gits/EnMP-banana.git {REPO_DIR}
!git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
!pip install -q -r requirements.txt
!apt-get -qq install -y fonts-noto-cjk > /dev/null

# 🐱 キャッシュ済みの古いモジュールを捨てて毎回最新を読み込む
for m in [k for k in list(sys.modules) if k.startswith("enmp_banana")]:
    del sys.modules[m]

## 2. 鍵とプレイリストURLを入れる

- Spotify: https://developer.spotify.com/dashboard で Create app → Client ID/Secret
- **Redirect URI** はDashboardで `http://127.0.0.1:8888/callback` を登録しておくこと（Authorization Code Flow用）
- Gemini: https://aistudio.google.com/apikey
- OpenAI: https://platform.openai.com/api-keys （`PROVIDER="openai"` を使うときだけ必要）

### Spotify認証について
このセルを最初に実行すると、Spotipyが認証用URLを表示します。

1. 表示されたURLをコピーしてブラウザで開く
2. Spotifyにログインして "Agree" を押す
3. リダイレクト先URL（`http://127.0.0.1:8888/callback?code=...` のような404ページのURLバー）を**まるごとコピー**
4. Colabのプロンプト `Enter the URL you were redirected to:` に貼り付けてEnter

`.spotify_cache` というファイルにtokenが保存され、2回目以降は自動で再利用されます（ランタイム再起動するまでは）。

In [ ]:
PLAYLIST_URL = "https://open.spotify.com/playlist/XXXXXXXXXXXXXXXXXXXXXX"

SPOTIFY_CLIENT_ID = ""
SPOTIFY_CLIENT_SECRET = ""
SPOTIFY_REDIRECT_URI = "http://127.0.0.1:8888/callback"  # 🐱 Dashboard登録値と完全一致させる

# 🐱 プロバイダ選択:
#   "collage"      = 無料・API不要、ジャケ写タイル+色抽出
#   "pollinations" = 無料・認証不要のAI画像生成(FLUX等)、生成30-120秒
#   "gemini"       = 有料tier必須、ジャケ写そのものをmix
#   "openai"       = 課金必要、gpt-image-1
PROVIDER = "pollinations"

GEMINI_API_KEY = ""
GEMINI_MODEL = "gemini-2.5-flash-image"

OPENAI_API_KEY = ""
OPENAI_MODEL = "gpt-image-1"

## 3. 生成

In [ ]:
from enmp_banana import make_thumbnail

img, meta = make_thumbnail(
    playlist_url=PLAYLIST_URL,
    spotify_client_id=SPOTIFY_CLIENT_ID,
    spotify_client_secret=SPOTIFY_CLIENT_SECRET,
    spotify_redirect_uri=SPOTIFY_REDIRECT_URI,
    provider=PROVIDER,
    gemini_api_key=GEMINI_API_KEY,
    gemini_model=GEMINI_MODEL,
    openai_api_key=OPENAI_API_KEY,
    openai_model=OPENAI_MODEL,
    output_path="playlist_thumbnail.png",
)

print(f"Playlist : {meta.name}")
print(f"Owner    : {meta.owner}")
print(f"Tracks   : {meta.track_count}")
print(f"Genres   : {meta.top_genres}")
img

## 4. ダウンロード

In [ ]:
from google.colab import files
files.download("playlist_thumbnail.png")